# Types Shared by the Notebooks of Chapters 4 and 5

In Python, every notebook of this chapter defines the types it needs, e.g. the type `RegExp` of regular
expressions is defined in eight different notebooks.  This is harmless in Python, since these types only
exist for the type checker.  In OCaml, every type definition creates a *new* type: if a notebook that defines
the type `regexp` is loaded twice, the values created with the first definition have a type that is different
from the second one, although both types look the same.  Therefore, the types used by more than one notebook
are defined in this notebook, which is loaded via `run_notebook "Types.ipynb"`.  Since `run_notebook` loads every
notebook only once, these types are only defined once.  The notebooks that use these types show their
definitions with the directive `#show_type`.

In [ ]:
#use "../lib/nb.ml";;

## Regular Expressions

A regular expression is represented by a value of the *variant type* `regexp`:
- `Empty` represents the regular expression $\emptyset$, which denotes the empty language,
- `Epsilon` represents the regular expression $\varepsilon$, which denotes the language $\{\varepsilon\}$,
- `Char c` represents the regular expression `c` that matches the character `c`,
- `Star r` represents $r^*$,
- `Product (r1, r2)` represents the concatenation $r_1 \cdot r_2$, and
- `Sum (r1, r2)` represents the union $r_1 + r_2$.

In Python, regular expressions are represented as nested tuples: $\emptyset$ is the number `0`, $\varepsilon$ is the
string `'𝜀'`, a character is a string of length 1, $r^*$ is the pair `(r, '*')`, and $r_1 \cdot r_2$ is the triple
`(r1, '⋅', r2)`.

**Type safety:** In the Python representation, the string `'𝜀'` could be mistaken for a character, the number `0`
for the character `'0'`, and a string like `'ab'` is admitted by the type checker although it is not a regular
expression.  In OCaml, every regular expression is built with exactly one of the six constructors, and every
`match` on a regular expression is checked for completeness by the compiler.

In [ ]:
type regexp =
  | Empty                         (* ∅       *)
  | Epsilon                       (* 𝜀       *)
  | Char    of char               (* c       *)
  | Star    of regexp             (* r*      *)
  | Product of regexp * regexp    (* r1 ⋅ r2 *)
  | Sum     of regexp * regexp    (* r1 + r2 *)

## Finite State Machines

The states of a finite state machine can be values of any type: the test notebooks use strings and
numbers, the function `nfa2dfa` produces machines whose states are *sets* of states, the function `minimize`
produces machines whose states are sets of states again, and the product automata of the notebook
`09-Equivalence.ipynb` have *pairs* of states.  Therefore, the types below are *polymorphic*: they have a
type parameter `'q`, which is the type of the states.  For example, `int nfa` is the type of the
non-deterministic finite state machines whose states are integers.

The transitions of a non-deterministic finite state machine are labelled either with a character or with
$\varepsilon$.  Python uses the string `'𝜀'` for the latter.  In OCaml, the labels have the type `label`: a label
is either `Eps` or `Sym c`, where `c` is a character.

A finite state machine is a quintuple $\langle Q, \Sigma, \delta, q_0, A \rangle$.  Python represents it as a tuple.  In
OCaml, we use a *record*, whose components are accessed by their names, e.g. `f.states`:
- `states` is the set $Q$ of states,
- `sigma` is the alphabet $\Sigma$,
- `delta` is the transition function $\delta$,
- `start` is the start state $q_0$, and
- `accepting` is the set $A$ of accepting states.

Sets are represented by the type `'a FSet.t` and dictionaries by the type `('k, 'v) Dict.t`, which are defined in
the file `../lib/fset.ml`.  The transition function of an *Nfa* maps a pair consisting of a state and a label
to a *set* of states, while the transition function of a *Dfa* maps a pair consisting of a state and a character to a
*single* state.

**Type safety:** Since *Nfa*s and *Dfa*s have different types, the compiler rejects an *Nfa* where a *Dfa* is
expected.  The Python notebooks need comments of the form `# type: ignore` at several places, because their
types are not precise enough.  None of these comments is needed in OCaml.

In [ ]:
type label = Eps | Sym of char

type 'q trans_rel     = ('q * label, 'q FSet.t) Dict.t    (* transition relation of an nfa *)
type 'q trans_rel_det = ('q * char, 'q) Dict.t            (* transition function of a dfa  *)

type 'q nfa = {
  states    : 'q FSet.t;
  sigma     : char FSet.t;
  delta     : 'q trans_rel;
  start     : 'q;
  accepting : 'q FSet.t;
}

type 'q dfa = {
  states    : 'q FSet.t;
  sigma     : char FSet.t;
  delta     : 'q trans_rel_det;
  start     : 'q;
  accepting : 'q FSet.t;
}

Since the types `nfa` and `dfa` have the same field names, the compiler uses the type of the context to find
out which of the two types is meant.  Therefore, we annotate the variables and functions whose values are
finite state machines with their types, e.g. `let f : int nfa = { ... }`.

Finally, the function `not_yet_defined` is used to initialize *forward references*, which are explained in the
notebook `RegExp-Parser.ipynb`.

In [ ]:
let not_yet_defined _ = failwith "not yet defined"